# LIBRARY

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [3]:
import pandas as pd

def analyze_feature(df, colonna, quantili=[0.1,0.25,0.50, 0.75,0.9,0.99]):
    serie = df[colonna] 
    
    # CAL values equal to 0
    num_zero = (serie == 0).sum()
    presenza_zero = num_zero > 0
    
    # CAL negative values
    num_negativi = (serie < 0).sum()
    presenza_negativi = num_negativi > 0
    
    # CAL range (min and max)
    valore_min = serie.min()
    valore_max = serie.max()
    range_valori = valore_max - valore_min

     # CAL quantiles
    valori_quantili = serie.quantile(quantili)
    
    # Stampa risultati
    print(f"Analyzed column: {colonna}")
    print(f"Presence of values equal to 0: {presenza_zero}")
    print(f"Number of values equal to 0: {num_zero}")
    print(f"Presence of negative values: {presenza_negativi}")
    print(f"Number of negative values: {num_negativi}")    
    print(f"Minimum value: {valore_min}")
    print(f"Maximum value: {valore_max}")
    print(f"Range (max - min): {range_valori}")

    print(f"\nQuantiles:")
    for q, v in valori_quantili.items():
        print(f"  Q{int(q*100)} ({q}): {v}")

# FEATURE SELECTION

## read df

In [4]:
file_path = "../1.DATASET/cmi_internet.csv"
df_clean=pd.read_csv(file_path)

In [5]:
df_clean=df_clean[['id',
    'Basic_Demos-Age', 
    'Basic_Demos-Sex',
    'CGAS-CGAS_Score', 
    'Physical-BMI',
    'Physical-Height', 
    'Physical-Weight',
    'Physical-Waist_Circumference',
    'Physical-Diastolic_BP',
    'Physical-HeartRate', 
    'Physical-Systolic_BP',
    'Fitness_Endurance-Max_Stage',
    'FGC-FGC_CU', 
    'FGC-FGC_GSND',
    'FGC-FGC_GSD', 
    'FGC-FGC_PU',
    'FGC-FGC_SRL', 
    'FGC-FGC_SRR',
    'FGC-FGC_TL', 
    'BIA-BIA_Activity_Level_num',
    'BIA-BIA_BMI',
    'BIA-BIA_BMC',
    'BIA-BIA_DEE',
    'BIA-BIA_ECW',
    'BIA-BIA_ICW',
    'BIA-BIA_FFMI', 
    'BIA-BIA_FMI', 
    'BIA-BIA_Fat',
    'BIA-BIA_Frame_num',
    'BIA-BIA_SMM',
    'BIA-BIA_TBW', 
    'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02',
    'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06',
    'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10',
    'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14',
    'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18',
    'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'PCIAT-PCIAT_Total', 
    'SDS-SDS_Total_T', 
    'PreInt_EduHx-computerinternet_hoursday', 'sii']]

## select features

### PCIAT-PCIAT_Total_CAL

In [6]:
cols_to_sum = ['PCIAT-PCIAT_01', 'PCIAT-PCIAT_02',
       'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06',
       'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10',
       'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14',
       'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18',
       'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20']
df_clean['PCIAT-PCIAT_Total_CAL']=df_clean[cols_to_sum].sum(axis=1)

df_clean[['PCIAT-PCIAT_Total','PCIAT-PCIAT_Total_CAL']][(round(df_clean['PCIAT-PCIAT_Total_CAL'],0)!=round(df_clean['PCIAT-PCIAT_Total'],0))
                                                        & (df_clean['PCIAT-PCIAT_Total_CAL']!=0)]

,PCIAT-PCIAT_Total,PCIAT-PCIAT_Total_CAL
24,30.0,31.0
141,26.0,27.0
255,81.0,82.0
270,48.0,49.0
425,5.0,7.0
944,53.0,54.0
1120,56.0,59.0
1247,40.0,41.0
1387,64.0,65.0
1472,74.0,75.0


In [7]:
analyze_feature(df_clean, 'PCIAT-PCIAT_Total_CAL')

Analyzed column: PCIAT-PCIAT_Total_CAL
Presence of values equal to 0: True
Number of values equal to 0: 6054
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 93.0
Range (max - min): 93.0

Quantiles:
  Q10 (0.1): 0.0
  Q25 (0.25): 0.0
  Q50 (0.5): 0.0
  Q75 (0.75): 9.0
  Q90 (0.9): 36.0
  Q99 (0.99): 71.0


### Mean Arterial Pressure (MAP)

In [8]:
df_clean['Physical-MAP_CAL'] = df_clean.apply(lambda x:
        x['Physical-Diastolic_BP']+((x['Physical-Systolic_BP']-x['Physical-Diastolic_BP'])/3)
        if x['Physical-Diastolic_BP']<=x['Physical-Systolic_BP'] 
        else x['Physical-Systolic_BP']+((x['Physical-Diastolic_BP']-x['Physical-Systolic_BP'])/3)
        , axis=1)

analyze_feature(df_clean, 'Physical-MAP_CAL')

Analyzed column: Physical-MAP_CAL
Presence of values equal to 0: True
Number of values equal to 0: 1
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 164.33333333333334
Range (max - min): 164.33333333333334

Quantiles:
  Q10 (0.1): 73.66666666666667
  Q25 (0.25): 78.5
  Q50 (0.5): 83.16666666666667
  Q75 (0.75): 88.83333333333333
  Q90 (0.9): 97.0
  Q99 (0.99): 122.33333333333333


### FGC-FGC_CU and FGC-FGC_PU

In [9]:
df_clean['FGC-FGC_CORE_CAL']=df_clean['FGC-FGC_CU']+df_clean['FGC-FGC_PU']
analyze_feature(df_clean, 'FGC-FGC_CORE_CAL')

Analyzed column: FGC-FGC_CORE_CAL
Presence of values equal to 0: True
Number of values equal to 0: 529
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 150.0
Range (max - min): 150.0

Quantiles:
  Q10 (0.1): 1.0
  Q25 (0.25): 5.0
  Q50 (0.5): 12.0
  Q75 (0.75): 20.0
  Q90 (0.9): 32.0
  Q99 (0.99): 62.0


### FGC-FGC_GSND and FGC-FGC_GSD

In [10]:
df_clean['FGC-FGC_GS_CAL']=(df_clean['FGC-FGC_GSND']+df_clean['FGC-FGC_GSD'])/2
analyze_feature(df_clean, 'FGC-FGC_GS_CAL')

Analyzed column: FGC-FGC_GS_CAL
Presence of values equal to 0: True
Number of values equal to 0: 3
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 115.1
Range (max - min): 115.1

Quantiles:
  Q10 (0.1): 13.975
  Q25 (0.25): 17.518749999999997
  Q50 (0.5): 18.875
  Q75 (0.75): 22.1
  Q90 (0.9): 29.475
  Q99 (0.99): 47.0065


### FGC-FGC_SRL and FGC-FGC_SRR

In [11]:
df_clean['FGC-FGC_SR_CAL']=(df_clean['FGC-FGC_SRR']+df_clean['FGC-FGC_SRL'])/2
analyze_feature(df_clean, 'FGC-FGC_SR_CAL')

Analyzed column: FGC-FGC_SR_CAL
Presence of values equal to 0: True
Number of values equal to 0: 87
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 21.0
Range (max - min): 21.0

Quantiles:
  Q10 (0.1): 5.0
  Q25 (0.25): 7.0
  Q50 (0.5): 8.5
  Q75 (0.75): 9.5
  Q90 (0.9): 11.25
  Q99 (0.99): 15.0


## DROP ATTRIBUITES

In [12]:
df_clean=df_clean.drop(columns = 
  [
      'Physical-Diastolic_BP','Physical-Systolic_BP', 'FGC-FGC_CU', 'FGC-FGC_GSND',
      'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR',
      'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02',
      'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06',
      'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10',
      'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14',
      'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18',
      'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'Physical-Waist_Circumference'

  ])

In [13]:
df_clean.shape

(8460, 31)

In [14]:
df_clean.describe()

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_TBW,PCIAT-PCIAT_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
count,8460.000000,8460.000000,8460.000000,7034.000000,7591.000000,7595.000000,7641.000000,7539.000000,5480.000000,6945.000000,...,6636.000000,2714.000000,4903.000000,7849.000000,8460.000000,8460.000000,6877.000000,5916.000000,4188.000000,5945.000000
mean,4229.500000,10.240189,0.402364,67.021041,19.532651,57.189069,84.428737,81.894150,4.918978,8.878107,...,51.694005,27.855195,58.072813,0.981144,0.443853,8.939598,84.606006,14.778736,20.624355,8.322321
std,2442.335972,3.574680,0.490404,35.284140,4.683925,7.374577,40.218441,11.380533,1.174191,2.558232,...,132.840059,20.318784,12.563287,1.046645,0.731014,17.370330,10.761730,13.785133,7.319546,2.606244
min,0.000000,5.000000,0.000000,25.000000,0.000000,33.000000,0.000000,27.000000,0.000000,0.000000,...,20.589200,0.000000,38.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2114.750000,7.000000,0.000000,60.500000,16.541742,51.500000,55.200000,75.500000,4.500000,8.000000,...,34.802217,12.000000,50.000000,0.000000,0.000000,0.000000,78.500000,5.000000,17.518750,7.000000
50%,4229.500000,10.000000,0.000000,65.000000,17.937682,55.700000,75.000000,81.500000,5.000000,9.000000,...,44.987000,26.000000,55.000000,1.000000,0.000000,0.000000,83.166667,12.000000,18.875000,8.500000
75%,6344.250000,12.000000,1.000000,70.500000,21.469546,63.000000,107.000000,87.000000,5.000000,10.000000,...,53.422779,41.000000,62.500000,2.000000,1.000000,9.000000,88.833333,20.000000,22.100000,9.500000
max,8459.000000,22.000000,1.000000,999.000000,59.132048,78.500000,315.000000,138.000000,28.000000,22.000000,...,5690.910000,93.000000,100.000000,3.000000,3.000000,93.000000,164.333333,150.000000,115.100000,21.000000


In [15]:
df_clean.columns

Index(['id', 'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-BMI', 'Physical-Height', 'Physical-Weight',
       'Physical-HeartRate', 'Fitness_Endurance-Max_Stage', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMI', 'BIA-BIA_BMC',
       'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_ICW', 'BIA-BIA_FFMI',
       'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_SMM',
       'BIA-BIA_TBW', 'PCIAT-PCIAT_Total', 'SDS-SDS_Total_T',
       'PreInt_EduHx-computerinternet_hoursday', 'sii',
       'PCIAT-PCIAT_Total_CAL', 'Physical-MAP_CAL', 'FGC-FGC_CORE_CAL',
       'FGC-FGC_GS_CAL', 'FGC-FGC_SR_CAL'],
      dtype='object')

# NULL VALUES

In [16]:
df_null=df_clean.isnull().sum()
df_null.sort_values()

id                                           0
Basic_Demos-Age                              0
Basic_Demos-Sex                              0
PCIAT-PCIAT_Total_CAL                        0
sii                                          0
PreInt_EduHx-computerinternet_hoursday     611
Physical-Weight                            819
Physical-Height                            865
Physical-BMI                               869
Physical-HeartRate                         921
CGAS-CGAS_Score                           1426
FGC-FGC_TL                                1515
Physical-MAP_CAL                          1583
BIA-BIA_TBW                               1824
BIA-BIA_SMM                               1824
BIA-BIA_Frame_num                         1824
BIA-BIA_Fat                               1824
BIA-BIA_ICW                               1824
BIA-BIA_FFMI                              1824
BIA-BIA_ECW                               1824
BIA-BIA_DEE                               1824
BIA-BIA_BMC  

## Physical-MAP_CAL'

In [17]:
df_MAP_CAL= df_clean[['Physical-MAP_CAL','Basic_Demos-Age']][df_clean['Physical-MAP_CAL'].notna()].groupby(['Basic_Demos-Age']).describe()
df_MAP_CAL=df_MAP_CAL.iloc[:, [1]].reset_index()
df_MAP_CAL.columns=['Basic_Demos-Age','mean_MAP_CAL']
df_MAP_CAL

,Basic_Demos-Age,mean_MAP_CAL
0,5,82.152821
1,6,81.940158
2,7,81.861037
3,8,82.926177
4,9,84.166427
5,10,84.630603
6,11,85.695946
7,12,84.767223
8,13,86.039162
9,14,88.446317


In [18]:
df_clean=df_clean.merge(df_MAP_CAL,on=['Basic_Demos-Age'],how="left")
df_clean[(df_clean['Physical-MAP_CAL'] == 0) | (df_clean['Physical-MAP_CAL'].isnull())]
df_clean.loc[((df_clean['Physical-MAP_CAL'] == 0) | (df_clean['Physical-MAP_CAL'].isnull())) , 'Physical-MAP_CAL'] = df_clean['mean_MAP_CAL']

In [19]:
df_clean[(df_clean['Physical-MAP_CAL'] == 0) | (df_clean['Physical-MAP_CAL'].isnull())]

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,PCIAT-PCIAT_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL,mean_MAP_CAL


In [20]:
analyze_feature(df_clean, 'Physical-MAP_CAL')

Analyzed column: Physical-MAP_CAL
Presence of values equal to 0: False
Number of values equal to 0: 0
Presence of negative values: False
Number of negative values: 0
Minimum value: 37.666666666666664
Maximum value: 164.33333333333334
Range (max - min): 126.66666666666669

Quantiles:
  Q10 (0.1): 74.66666666666667
  Q25 (0.25): 79.79166666666666
  Q50 (0.5): 83.33333333333333
  Q75 (0.75): 88.16666666666667
  Q90 (0.9): 95.16666666666667
  Q99 (0.99): 119.66666666666667


## 'FGC-FGC_CORE_CAL'

In [21]:
df_FGC_CORE_CAL= df_clean[['FGC-FGC_CORE_CAL','Basic_Demos-Age']][df_clean['FGC-FGC_CORE_CAL'].notna()].groupby(['Basic_Demos-Age']).describe()
df_FGC_CORE_CAL=df_FGC_CORE_CAL.iloc[:, [1]].reset_index()
df_FGC_CORE_CAL.columns=['Basic_Demos-Age','mean_CORE_CAL']
df_FGC_CORE_CAL

,Basic_Demos-Age,mean_CORE_CAL
0,5,4.611111
1,6,6.280570
2,7,9.540221
3,8,10.568299
4,9,12.787145
5,10,14.662915
6,11,18.198142
7,12,19.750620
8,13,21.493846
9,14,24.748963


In [22]:
df_clean=df_clean.merge(df_FGC_CORE_CAL,on=['Basic_Demos-Age'],how="left")
df_clean[(df_clean['FGC-FGC_CORE_CAL'] == 0) | (df_clean['FGC-FGC_CORE_CAL'].isnull())]
df_clean.loc[((df_clean['FGC-FGC_CORE_CAL'] == 0) | (df_clean['FGC-FGC_CORE_CAL'].isnull())) , 'FGC-FGC_CORE_CAL'] = df_clean['mean_CORE_CAL']

In [23]:
df_clean[(df_clean['FGC-FGC_CORE_CAL'] == 0) | (df_clean['FGC-FGC_CORE_CAL'].isnull())]

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL,mean_MAP_CAL,mean_CORE_CAL


In [24]:
analyze_feature(df_clean, 'FGC-FGC_CORE_CAL')

Analyzed column: FGC-FGC_CORE_CAL
Presence of values equal to 0: False
Number of values equal to 0: 0
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.5
Maximum value: 150.0
Range (max - min): 149.5

Quantiles:
  Q10 (0.1): 4.5
  Q25 (0.25): 8.0
  Q50 (0.5): 12.787145242070117
  Q75 (0.75): 20.5
  Q90 (0.9): 28.0
  Q99 (0.99): 57.0


## 'FGC-FGC_GS_CAL'

In [25]:
df_FGC_GS_CAL= df_clean[['FGC-FGC_GS_CAL','Basic_Demos-Age']][df_clean['FGC-FGC_GS_CAL'].notna()].groupby(['Basic_Demos-Age']).describe()
df_FGC_GS_CAL=df_FGC_GS_CAL.iloc[:, [1]].reset_index()
df_FGC_GS_CAL.columns=['Basic_Demos-Age','mean_GS_CAL']
df_FGC_GS_CAL

,Basic_Demos-Age,mean_GS_CAL
0,5,17.883041
1,6,18.613018
2,7,18.909726
3,8,18.526452
4,9,18.204545
5,10,16.840263
6,11,18.544625
7,12,20.626092
8,13,22.080423
9,14,26.124128


In [26]:
df_clean=df_clean.merge(df_FGC_GS_CAL,on=['Basic_Demos-Age'],how="left")
df_clean[(df_clean['FGC-FGC_GS_CAL'] == 0) | (df_clean['FGC-FGC_GS_CAL'].isnull())]
df_clean.loc[((df_clean['FGC-FGC_GS_CAL'] == 0) | (df_clean['FGC-FGC_GS_CAL'].isnull())) , 'FGC-FGC_GS_CAL'] = df_clean['mean_GS_CAL']

In [27]:
df_clean[(df_clean['FGC-FGC_GS_CAL'] == 0) | (df_clean['FGC-FGC_GS_CAL'].isnull())]

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL,mean_MAP_CAL,mean_CORE_CAL,mean_GS_CAL


In [28]:
analyze_feature(df_clean, 'FGC-FGC_GS_CAL')

Analyzed column: FGC-FGC_GS_CAL
Presence of values equal to 0: False
Number of values equal to 0: 0
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.1
Maximum value: 115.1
Range (max - min): 115.0

Quantiles:
  Q10 (0.1): 16.3
  Q25 (0.25): 18.025
  Q50 (0.5): 18.61301775147929
  Q75 (0.75): 20.626091644204852
  Q90 (0.9): 26.714086956521736
  Q99 (0.99): 40.296749999999975


## 'FGC-FGC_SR_CAL'

In [29]:
df_FGC_SR_CAL= df_clean[['FGC-FGC_SR_CAL','Basic_Demos-Age']][df_clean['FGC-FGC_SR_CAL'].notna()].groupby(['Basic_Demos-Age']).describe()
df_FGC_SR_CAL=df_FGC_SR_CAL.iloc[:, [1]].reset_index()
df_FGC_SR_CAL.columns=['Basic_Demos-Age','mean_SR_CAL']
df_FGC_SR_CAL

,Basic_Demos-Age,mean_SR_CAL
0,5,8.897552
1,6,8.886824
2,7,8.749076
3,8,8.667797
4,9,8.492420
5,10,7.682875
6,11,8.123333
7,12,7.529942
8,13,7.426384
9,14,8.556818


In [30]:
df_clean=df_clean.merge(df_FGC_SR_CAL,on=['Basic_Demos-Age'],how="left")
df_clean[(df_clean['FGC-FGC_SR_CAL'] == 0) | (df_clean['FGC-FGC_SR_CAL'].isnull())]
df_clean.loc[((df_clean['FGC-FGC_SR_CAL'] == 0) | (df_clean['FGC-FGC_SR_CAL'].isnull())) , 'FGC-FGC_SR_CAL'] = df_clean['mean_SR_CAL']

In [31]:
df_clean[(df_clean['FGC-FGC_SR_CAL'] == 0) | (df_clean['FGC-FGC_SR_CAL'].isnull())]

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL,mean_MAP_CAL,mean_CORE_CAL,mean_GS_CAL,mean_SR_CAL
997,997,22,0,NaN,17.937682,55.00,NaN,81.0,NaN,10.0,...,0.0,0.0,89.791667,21.25,25.191667,NaN,89.791667,21.25,25.191667,NaN
1282,1282,22,1,68.0,17.937682,55.00,77.0,NaN,5.0,10.0,...,0.0,0.0,83.333333,21.25,25.191667,NaN,89.791667,21.25,25.191667,NaN
1298,1298,22,1,65.0,34.509396,65.00,207.4,NaN,5.0,NaN,...,1.0,44.0,83.333333,12.00,20.625000,NaN,89.791667,21.25,25.191667,NaN
1522,1522,22,1,65.0,24.835584,62.50,138.0,71.0,5.0,NaN,...,0.0,0.0,93.333333,21.25,25.191667,NaN,89.791667,21.25,25.191667,NaN
1602,1602,22,1,NaN,22.707130,63.00,128.2,81.0,NaN,10.0,...,0.0,0.0,86.333333,21.25,20.625000,NaN,89.791667,21.25,25.191667,NaN
2048,2048,22,1,NaN,23.657232,64.50,140.0,69.0,NaN,NaN,...,0.0,0.0,82.000000,21.25,25.191667,NaN,89.791667,21.25,25.191667,NaN
2345,2345,22,1,75.0,26.056757,65.00,156.6,78.0,NaN,10.0,...,0.0,0.0,96.333333,21.25,25.191667,NaN,89.791667,21.25,25.191667,NaN
3297,3297,22,1,NaN,40.898793,65.00,245.8,120.0,NaN,NaN,...,1.0,31.0,111.333333,21.25,25.191667,NaN,89.791667,21.25,25.191667,NaN
8348,8348,22,1,485.5,23.654948,64.97,119.5,72.5,4.5,10.0,...,1.0,0.0,82.333333,30.50,34.325000,NaN,89.791667,21.25,25.191667,NaN


In [32]:
analyze_feature(df_clean, 'FGC-FGC_SR_CAL')

Analyzed column: FGC-FGC_SR_CAL
Presence of values equal to 0: False
Number of values equal to 0: 0
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.5
Maximum value: 21.0
Range (max - min): 20.5

Quantiles:
  Q10 (0.1): 6.0
  Q25 (0.25): 7.5
  Q50 (0.5): 8.5
  Q75 (0.75): 9.0
  Q90 (0.9): 10.5
  Q99 (0.99): 14.5


## DROP ATTRIBUITES

In [33]:
df_clean=df_clean.drop(columns = 
  [
      'mean_MAP_CAL', 'mean_CORE_CAL',
       'mean_GS_CAL', 'mean_SR_CAL', 'PCIAT-PCIAT_Total_CAL','PCIAT-PCIAT_Total'

  ])

In [34]:
df_clean.columns

Index(['id', 'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-BMI', 'Physical-Height', 'Physical-Weight',
       'Physical-HeartRate', 'Fitness_Endurance-Max_Stage', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMI', 'BIA-BIA_BMC',
       'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_ICW', 'BIA-BIA_FFMI',
       'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_SMM',
       'BIA-BIA_TBW', 'SDS-SDS_Total_T',
       'PreInt_EduHx-computerinternet_hoursday', 'sii', 'Physical-MAP_CAL',
       'FGC-FGC_CORE_CAL', 'FGC-FGC_GS_CAL', 'FGC-FGC_SR_CAL'],
      dtype='object')

## VALIDATION

In [35]:
df_null=df_clean.isnull().sum()
df_null.sort_values()

id                                           0
Basic_Demos-Age                              0
Basic_Demos-Sex                              0
FGC-FGC_CORE_CAL                             0
Physical-MAP_CAL                             0
sii                                          0
FGC-FGC_GS_CAL                               0
FGC-FGC_SR_CAL                               9
PreInt_EduHx-computerinternet_hoursday     611
Physical-Weight                            819
Physical-Height                            865
Physical-BMI                               869
Physical-HeartRate                         921
CGAS-CGAS_Score                           1426
FGC-FGC_TL                                1515
BIA-BIA_Fat                               1824
BIA-BIA_TBW                               1824
BIA-BIA_SMM                               1824
BIA-BIA_Frame_num                         1824
BIA-BIA_ECW                               1824
BIA-BIA_FFMI                              1824
BIA-BIA_ICW  

# CONTROL DATASET 

In [36]:
file_path = "../1.DATASET/control_id.csv"
df_id=pd.read_csv(file_path)

In [37]:
df_train=df_clean.copy()
df_train

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_Frame_num,BIA-BIA_SMM,BIA-BIA_TBW,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
0,0,5,0,51.0,16.877316,46.00,50.8,NaN,5.0,6.0,...,1.0,19.541300,32.690900,NaN,3.0,2.0,82.152821,4.611111,17.883041,6.500000
1,1,9,0,NaN,14.035590,48.00,46.0,70.0,NaN,3.0,...,1.0,15.410700,27.055200,64.0,0.0,0.0,90.666667,8.000000,18.204545,11.000000
2,2,10,1,71.0,16.648696,56.50,75.6,94.0,5.0,5.0,...,NaN,NaN,44.987000,54.0,2.0,0.0,82.333333,27.000000,12.450000,10.000000
3,3,9,0,71.0,18.292347,56.00,81.6,97.0,6.0,7.0,...,2.0,26.479800,45.996600,45.0,0.0,1.0,79.000000,23.000000,18.204545,7.000000
4,4,18,1,65.0,17.937682,NaN,77.0,NaN,NaN,10.0,...,2.0,NaN,NaN,NaN,1.0,0.0,90.152104,12.000000,26.463732,8.003205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8455,8455,7,1,NaN,16.130585,46.07,49.0,82.5,2.5,8.0,...,1.0,NaN,31.583004,55.0,0.0,0.0,81.861037,6.500000,18.075000,7.000000
8456,8456,10,1,69.5,NaN,56.13,47.8,80.5,5.0,8.0,...,1.0,22.983200,36.475662,NaN,0.0,1.0,79.833333,8.500000,13.575000,5.000000
8457,8457,10,1,70.0,40.937571,49.56,47.2,83.5,7.0,NaN,...,2.0,31.594712,35.106855,NaN,2.0,0.0,82.166667,21.500000,16.840263,9.500000
8458,8458,15,1,55.5,NaN,63.79,99.5,87.5,NaN,9.0,...,2.0,46.467581,57.530686,NaN,1.0,2.0,88.395702,10.000000,26.417470,10.500000


In [38]:
df_train_control = pd.merge(df_train, df_id, on=['id'], how='inner')
df_train_control

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_Frame_num,BIA-BIA_SMM,BIA-BIA_TBW,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
0,1,9,0,NaN,14.035590,48.00,46.0,70.0,NaN,3.0,...,1.0,15.4107,27.0552,64.0,0.0,0.0,90.666667,8.000000,18.204545,11.000000
1,3,9,0,71.0,18.292347,56.00,81.6,97.0,6.0,7.0,...,2.0,26.4798,45.9966,45.0,0.0,1.0,79.000000,23.000000,18.204545,7.000000
2,5,13,1,50.0,22.279952,59.50,112.2,73.0,5.0,8.0,...,2.0,35.3804,63.1265,56.0,0.0,1.0,74.000000,18.000000,17.200000,10.500000
3,6,10,0,NaN,19.660760,55.00,84.6,83.0,NaN,11.0,...,2.0,26.1957,47.2211,40.0,3.0,0.0,136.333333,11.000000,16.840263,11.000000
4,7,10,1,65.0,16.861286,59.25,84.2,90.0,5.0,4.0,...,2.0,28.7680,50.4767,55.0,2.0,0.0,86.000000,14.662915,11.850000,7.682875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852,3952,15,0,40.0,26.364710,70.50,186.4,74.0,5.0,10.0,...,3.0,54.2027,91.4887,90.0,3.0,1.0,117.333333,23.714789,26.417470,8.527654
1853,3953,8,0,65.0,17.139810,52.50,67.2,65.0,NaN,12.0,...,1.0,20.2645,36.7181,58.0,2.0,0.0,77.333333,10.568299,18.526452,9.000000
1854,3954,7,1,NaN,13.927006,48.50,46.6,75.0,5.0,4.5,...,1.0,18.0937,30.0453,67.0,0.0,1.0,78.333333,9.540221,18.909726,8.750000
1855,3955,13,0,60.0,16.362460,59.50,82.4,70.0,NaN,12.0,...,1.0,29.7790,52.8320,50.0,1.0,1.0,82.000000,26.000000,18.950000,8.500000


In [39]:
df_train_control.to_csv('../1.DATASET/CMI_V2_CONTROL.csv', index=False)

### check values 

In [40]:
df_train_control.shape

(1857, 29)

In [41]:
df_train_control.columns

Index(['id', 'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-BMI', 'Physical-Height', 'Physical-Weight',
       'Physical-HeartRate', 'Fitness_Endurance-Max_Stage', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMI', 'BIA-BIA_BMC',
       'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_ICW', 'BIA-BIA_FFMI',
       'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_SMM',
       'BIA-BIA_TBW', 'SDS-SDS_Total_T',
       'PreInt_EduHx-computerinternet_hoursday', 'sii', 'Physical-MAP_CAL',
       'FGC-FGC_CORE_CAL', 'FGC-FGC_GS_CAL', 'FGC-FGC_SR_CAL'],
      dtype='object')

In [42]:
df_train_control[
    (df_train_control["Physical-Weight"] < 0)
    #& (df_train_control["BIA-BIA_SMM"] > df_train_control["Physical-Weight"])   
    #& (df_train_control["BIA-BIA_Fat"] > df_train_control["Physical-Weight"])
    #& (df_train_control['BIA-BIA_TBW'] > df_train_control["Physical-Weight"])
    #& (df_train_control['BIA-BIA_TBW'] > df_train_control["Physical-Weight"])
    #& (df_train_control['BIA-BIA_BMC']<=0) #yes 
    #& (df_train_control['BIA-BIA_FMI']<=0) # yes
    #& (df_train_control['BIA-BIA_Fat']<=0) #yes
    
    
]

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_Frame_num,BIA-BIA_SMM,BIA-BIA_TBW,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL


# FULL DATASET 

In [43]:
df_clean.shape

(8460, 29)

In [45]:
df_clean

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_Frame_num,BIA-BIA_SMM,BIA-BIA_TBW,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
0,0,5,0,51.0,16.877316,46.00,50.8,NaN,5.0,6.0,...,1.0,19.541300,32.690900,NaN,3.0,2.0,82.152821,4.611111,17.883041,6.500000
1,1,9,0,NaN,14.035590,48.00,46.0,70.0,NaN,3.0,...,1.0,15.410700,27.055200,64.0,0.0,0.0,90.666667,8.000000,18.204545,11.000000
2,2,10,1,71.0,16.648696,56.50,75.6,94.0,5.0,5.0,...,NaN,NaN,44.987000,54.0,2.0,0.0,82.333333,27.000000,12.450000,10.000000
3,3,9,0,71.0,18.292347,56.00,81.6,97.0,6.0,7.0,...,2.0,26.479800,45.996600,45.0,0.0,1.0,79.000000,23.000000,18.204545,7.000000
4,4,18,1,65.0,17.937682,NaN,77.0,NaN,NaN,10.0,...,2.0,NaN,NaN,NaN,1.0,0.0,90.152104,12.000000,26.463732,8.003205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8455,8455,7,1,NaN,16.130585,46.07,49.0,82.5,2.5,8.0,...,1.0,NaN,31.583004,55.0,0.0,0.0,81.861037,6.500000,18.075000,7.000000
8456,8456,10,1,69.5,NaN,56.13,47.8,80.5,5.0,8.0,...,1.0,22.983200,36.475662,NaN,0.0,1.0,79.833333,8.500000,13.575000,5.000000
8457,8457,10,1,70.0,40.937571,49.56,47.2,83.5,7.0,NaN,...,2.0,31.594712,35.106855,NaN,2.0,0.0,82.166667,21.500000,16.840263,9.500000
8458,8458,15,1,55.5,NaN,63.79,99.5,87.5,NaN,9.0,...,2.0,46.467581,57.530686,NaN,1.0,2.0,88.395702,10.000000,26.417470,10.500000


In [48]:
df_clean.isnull().sum().sort_values()

id                                           0
Basic_Demos-Age                              0
Basic_Demos-Sex                              0
FGC-FGC_CORE_CAL                             0
Physical-MAP_CAL                             0
sii                                          0
FGC-FGC_GS_CAL                               0
FGC-FGC_SR_CAL                               9
PreInt_EduHx-computerinternet_hoursday     611
Physical-Weight                            819
Physical-Height                            865
Physical-BMI                               869
Physical-HeartRate                         921
CGAS-CGAS_Score                           1426
FGC-FGC_TL                                1515
BIA-BIA_Fat                               1824
BIA-BIA_TBW                               1824
BIA-BIA_SMM                               1824
BIA-BIA_Frame_num                         1824
BIA-BIA_ECW                               1824
BIA-BIA_FFMI                              1824
BIA-BIA_ICW  

In [49]:
df_clean.to_csv('../1.DATASET/CMI_V2.csv', index=False)